# **TWIN760**

In [1]:
import pandas as pd
import numpy as np
import twinlab as tl


          ====== TwinLab Client Initialisation ======
          Version     : 2.4.0
          Server      : http://127.0.0.1:3000
          Environment : /Users/sergiochavez/twinlab-demos/.env



## **I. Fidelity in SampleParams**


In [2]:
# Create dataset and train model
df = tl.load_dataset("multi_fidelity.csv")
dataset = tl.Dataset(id="multi_fidelity")
dataset.upload(df)

# Create and train emulator
estimator_params = tl.EstimatorParams(estimator_type="multi_fidelity_gp")
params = tl.TrainParams(
        train_test_ratio=0.8,
        estimator_params=estimator_params,
        seed=123,
        fidelity="fidelity",
    )
emulator = tl.Emulator(id="multi_fidelity")
emulator.train(dataset, inputs=["x"], outputs=["y"], params=params)

In [4]:
# The required dataframe with the weights
sampling_df = pd.DataFrame({"fidelity": np.hstack((np.ones(5), 0.5 * np.ones(15)))})
# Sampling
sample_params = tl.SampleParams(
        seed=123,
        fidelity=sampling_df) #This used to throw the error: object not serializable
X = np.linspace(0, 1, 20)
input_df = pd.DataFrame({"x": X})
sample_df = emulator.sample(input_df, num_samples=1, params=sample_params) 

In [5]:
#Classic sample to compare
classic_sample = emulator.sample(input_df, num_samples=1)

In [6]:
comparison=pd.concat([sampling_df,sample_df, classic_sample], axis=1)
comparison.columns = ["Weights (fidelity)", "Sample with fidelity", "Sample without fidelity"]
comparison

,Weights (fidelity),Sample with fidelity,Sample without fidelity
0,1.0,3.452415,3.112462
1,1.0,0.041438,1.145462
2,1.0,-0.891242,-0.271806
3,1.0,-0.776771,-1.206939
4,1.0,-0.776290,-0.368280
5,0.5,-0.328494,-0.260287
6,0.5,-0.242945,0.546808
7,0.5,0.086493,0.794957
8,0.5,-0.032682,0.746597
9,0.5,0.867378,0.049377


## **II. bounds in OptimiserParams**

In [7]:
# Create dataset (this comes from the example in the documentation)
np.random.seed(123)
def f(x, a=6, b=12):
    return (a * x - 2) ** 2 * np.sin(b * x - 4)


X = np.linspace(0, 1, 100)[:, np.newaxis]
y = f(X)  # Arrange outputs as feature columns

# Set up training data dataframe
X_data = np.array([0.12, 0.311, 0.409, 0.502, 0.598, 0.921, 0.9765])
y_data = []
for i in X_data:
    y_data.append(f(i) + np.random.normal(scale=0.1))
# Convert to DataFrame
df_train = pd.DataFrame({"X": X_data, "y": y_data})
df_test = pd.DataFrame({"X": X.flatten(), "y": y.flatten()})

# Define the name of the dataset
dataset_id = "Training_Data"

dataset_1D = tl.Dataset(id=dataset_id)

# Upload the dataset to the cloud
dataset_1D.upload(df_train, verbose=True)

Dataframe is uploading.
Processing dataset
Dataset Training_Data was processed.


In [8]:
# Initialise emulator
emulator_id = "basic_emulator"

emulator = tl.Emulator(id=emulator_id)

# Train the emulator using the train method
emulator.train(dataset=dataset_1D, inputs=["X"], outputs=["y"], verbose=True)

Model basic_emulator has begun training.
Emulator basic_emulator with process ID 87acdaac-a287-4cb6-849b-81de6a07d002 is training.
Training status: processing_Modal
Training status: done_Modal
Training of emulator basic_emulator with process ID 87acdaac-a287-4cb6-849b-81de6a07d002 is complete!


In [9]:
# Classic recommend
emulator.recommend(num_points=1, acq_func="global")

(         X
 0  0.76371,
 2.2106473243797464)

In [10]:
# Recommend with bounds
bound_df=pd.DataFrame({"X": [0.3,0.8]},index=["lower_bounds","upper_bounds"]) # The df requires this format
optimiser_params=tl.OptimiserParams(bounds=bound_df)
recommend_params=tl.RecommendParams(opt_kwargs=optimiser_params,seed=123)
emulator.recommend(num_points=2, acq_func="global",params=recommend_params)

(          X
 0  0.688049
 1  0.755920,
 -1.050248914955382)

## **III. y_std_model in CalibrateParams**


In [12]:
# Data and model parameters (this comes from notebook number 11 in main branch in demos)
random_seed = 8675309
err_sig = 2/3
# Seed the random-number generator
np.random.seed(random_seed)

# Create a linear function to generate experimental data
def f(x):
    return 2 * x + 3

# Add Gaussian noise to the linear ouput
def model(X):
    return np.random.normal(f(X), err_sig)

# Set up experimental data dataframe
X = np.linspace(0,10,21)
y = model(X)
df_exp = pd.DataFrame({"X": X, "y": y})
display(df_exp)

# Upload the dataset to the cloud
dataset = tl.Dataset("inverse-methods-dataset")
dataset.upload(df_exp)

# Train model
emulator = tl.Emulator("inverse-methods-model")
emulator.train(dataset, ['X'], ['y'])

,X,y
0,0.0,3.392682
1,0.5,4.488746
2,1.0,4.225207
3,1.5,5.628789
4,2.0,6.485008
5,2.5,7.887852
6,3.0,8.722331
7,3.5,9.081046
8,4.0,11.499504
9,4.5,12.119256


In [14]:
# Observed datapoint we would like to know the inputs for.
obs = pd.DataFrame({'y': [10.25]})
display(obs)
obs_std = pd.DataFrame({'y': [err_sig]})
display(obs_std)
# This is the required argument for CalibrateParams
noise_df = pd.DataFrame({'y': [1/9]})
display(noise_df)

,y
0,10.25


,y
0,0.666667


,y
0,0.111111


In [17]:
# Classic calibrate. Solve the inverse problem (Without calibrate params noise df)
inverse_df = emulator.calibrate(df_obs=obs, df_std=obs_std)
inverse_df

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
X,3.724,0.335,3.089,4.35,0.007,0.005,2240.0,2640.0,1.0


In [18]:
# With fix
calibrate_params=tl.CalibrateParams(y_std_model=noise_df)
inverse_df2 = emulator.calibrate(df_obs=obs, df_std=obs_std, params=calibrate_params)
inverse_df2

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
X,3.734,0.421,2.943,4.518,0.009,0.007,2091.0,2856.0,1.0
